### RAG PIPILANES - DATA INGESTION TO VECTOR DB PIPELAIN

In [10]:
%pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.


c:\Users\HP\Documents\works\RAG test\.venv\Scripts\python.exe: No module named pip


In [1]:
import sys

print(sys.executable)

c:\Users\HP\Documents\works\RAG test\.venv\Scripts\python.exe


In [2]:
import langchain_ollama

print("langchain_ollama installed successfully!")

c:\Users\HP\Documents\works\RAG test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


langchain_ollama installed successfully!


In [3]:
import sys

print("Python used by Jupyter:")
print(sys.executable)

Python used by Jupyter:
c:\Users\HP\Documents\works\RAG test\.venv\Scripts\python.exe


In [4]:
import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "langchain-ollama"
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\n--- STDOUT ---")
print(result.stdout)
print("\n--- STDERR ---")
print(result.stderr)

RETURN CODE: 0

--- STDOUT ---


--- STDERR ---

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



In [5]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "ensurepip",
    "--upgrade"
])

0

In [6]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "langchain-ollama"
])

0

In [7]:
from langchain_ollama import OllamaEmbeddings, ChatOllama

print("SUCCESS!")

SUCCESS!


In [8]:
import os
import numpy as np
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from pathlib import Path

In [27]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in the directory."""

    all_documents = []

    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:

        print(f"\nprocessing: {pdf_file.name}")

        try:
            # Use PyMuPDF instead of PyPDF
            loader = PyMuPDFLoader(str(pdf_file))

            documents = loader.load()

            for doc in documents:

                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(
                f"loaded {len(documents)} pages"
            )

        except Exception as e:

            print(
                f"ERROR processing {pdf_file.name}: {e}"
            )

    print(
        f"\ntotal documents loaded: "
        f"{len(all_documents)}"
    )

    return all_documents


all_pdf_documents = process_all_pdfs("../data")

found 2 PDF files to process

processing: doctorfrankel.pdf
loaded 48 pages

processing: markmensen.Pdf
loaded 148 pages

total documents loaded: 196


In [28]:
all_pdf_documents

[Document(metadata={'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'source': '..\\data\\pdf\\doctorfrankel.pdf', 'file_path': '..\\data\\pdf\\doctorfrankel.pdf', 'total_pages': 48, 'format': 'PDF 1.4', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20110912133108+04'30'", 'page': 0, 'source_file': 'doctorfrankel.pdf', 'file_type': 'pdf'}, page_content='١  \naspx\n.\n151\n-\npost\n/\ncom\n.\nblogfa\n.\naarmaan\n://\nhttp\n  \n  \nا ﻧﺴﺎن در ﺟﺴﺘﺠﻮي ﻣﻌﻨﯽ \n ﻧﻮﺷﺘﻪ\n: \nدﮐﺘﺮ وﯾﮑﺘﻮر ﻓﺮاﻧﮑﻞ \n  \nاﺳﺘﺎد روان ﭘﺰﺷﮑﯽ داﻧﺸﮕﺎه وﯾﻦ \n  \nﺗﺮﺟﻤﻪ \n: \nدﮐﺘﺮ اﮐﺒﺮ ﻣﻌﺎرﻓﯽ \n  \n  \n \nﺗﻮﺿ\nﯿ\nﺤﺎت  ﻣﺘﺮﺟﻢ ﻓﺎرﺳﯽ  \nﺑﺎ ﭘﯿﺶ درآﻣﺪي ﮐﻪ دﮐﺘﺮ ﮔﻮردو\nن آﻟﭙﻮرت اﺳﺘﺎد ﭘﯿﺸﯿﻦ رواﻧﺸﻨﺎﺳﯽ داﻧﺸﮕﺎه ﻫﺎروارد ﺑﺮ اﯾﻦ ﮐﺘﺎب ﻧﻮﺷﺘﻪ اﺳﺖ ﺟﺎ ﻧﺪارد ﻣﻘﺪﻣﻪ \nدراز دﯾﮕﺮي ﻧﻮﺷﺘﻪ ﺷﻮد و ﻣﻦ ﺗﻨﻬﺎ ﺗﻮﺿﯿﺤﺎﺗﯽ را ﻣ\nﯽ \nاﻓﺰاﯾﻢ ﮐﻪ در آن ﻧﻮﺷﺘﻪ ﻧ

In [11]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """ split documents into smaller chunks for better RAG performance"""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )
    split_doc = text_splitter.split_documents(documents)
    print(f"split{len(documents)} documents into {len(split_doc)}chunks")

    if split_doc:
        print(f"\nexample:")
        print(f"content:{split_doc[0].page_content[:200]}....")
        print(f"metadata:{split_doc[0].metadata}")
    return split_doc

In [12]:
chunks = split_documents(all_pdf_documents)
chunks

split196 documents into 384chunks

example:
content:١ 
aspx.151-post/com.blogfa.aarmaan://http  
  
ا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن 
 ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  
 وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  
 ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  
  
 
ﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت 
ﮔﻮردو دﮐﺘﺮ ....
metadata:{'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'source': '..\\data\\pdf\\doctorfrankel.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'doctorfrankel.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'source': '..\\data\\pdf\\doctorfrankel.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'doctorfrankel.pdf', 'file_type': 'pdf'}, page_content='١ \naspx.151-post/com.blogfa.aarmaan://http  \n  \nا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن \n ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  \n وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  \n ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  \n  \n \nﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت \nﮔﻮردو دﮐﺘﺮ ﮐﻪ درآﻣﺪي ﭘﯿﺶ ﺑﺎ ﻣﻘﺪﻣﻪ ﻧﺪارد ﺟﺎ اﺳﺖ ﻧﻮﺷﺘﻪ ﮐﺘﺎب اﯾﻦ ﺑﺮ ﻫﺎروارد داﻧﺸﮕﺎه رواﻧﺸﻨﺎﺳﯽ ﭘﯿﺸﯿﻦ اﺳﺘﺎد آﻟﭙﻮرت ن\nﻣ را ﺗﻮﺿﯿﺤﺎﺗﯽ ﺗﻨﻬﺎ ﻣﻦ و ﺷﻮد ﻧﻮﺷﺘﻪ دﯾﮕﺮي ﯽدراز  ﻧﯿﺴﺖ ﻧﻮﺷﺘﻪ آن در ﮐﻪ اﻓﺰاﯾﻢ  . \n ﯾﻌﻨﯽ ﮐﺘﺎب اﯾﻦ اول اﺳﯿﺮان اردوي در ﺳﺮﮔﺸﺘﯽ «   را آن ﻣـﻦ و اﺳـﺖ ﻧﻮﺷﺘﻪ آﻟﻤﺎﻧﯽ ﺑﺰﺑﺎن ﻓﺮاﻧﮑﻞ دﮐﺘﺮ را  ﺗﺮﺟﻤـﻪ از\nام درآورده ﺑﻔﺎرﺳﯽ ﻻش اﯾﻠﺰه اﻧﮕﻠﯿﺴﯽ . ﯾﻌﻨﯽ دﯾﮕﺮ ﺑﺨﺶ ﻟﻮﮔﻮﺗﺮاﭘﯽ اﺳﺎﺳﯽ ﻣﻔﻬﻮم اﺳﺖ ﻧﻮﺷﺘﻪ ﺑﺎﻧﮕﻠﯿﺴﯽ ﻓﺮاﻧﮑﻞ را 

### EMBEDING AND VECTORSTOREDB

In [13]:

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List , Dict , Any ,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingManager:
    """
    Handles document and query embeddings using
    Ollama's local embedding model.
    """

    def __init__(self, model_name: str = "nomic-embed-text:latest"):
        self.model_name = model_name

        print(f"loading Ollama embedding model: {self.model_name}")

        self.model = OllamaEmbeddings(
            model=self.model_name
        )

        # Test connection
        test_embedding = self.model.embed_query("test")

        print(
            "Ollama embedding model loaded successfully!"
        )

        print(
            f"embedding dimension: {len(test_embedding)}"
        )

    def generate_embedding(self, texts: list[str]) -> np.ndarray:
        """
        Generate embeddings for multiple documents.
        """

        if not texts:
            return np.array([])

        print(
            f"generating Ollama embeddings for "
            f"{len(texts)} texts..."
        )

        embeddings = self.model.embed_documents(texts)

        embeddings = np.array(
            embeddings,
            dtype=np.float32
        )

        print(
            f"generated embeddings with shape: "
            f"{embeddings.shape}"
        )

        return embeddings

    def generate_query_embedding(self, query: str) -> np.ndarray:
        """
        Generate embedding for a single query.
        """

        embedding = self.model.embed_query(query)

        return np.array(
            embedding,
            dtype=np.float32
        )

    def get_embedding_dimension(self) -> int:
        """
        Return embedding dimension.
        """

        embedding = self.model.embed_query("test")

        return len(embedding)


embedding_manager = EmbeddingManager(
    model_name="nomic-embed-text:latest"
)

loading Ollama embedding model: nomic-embed-text:latest
Ollama embedding model loaded successfully!
embedding dimension: 768


### VECTOR STORE 

In [15]:

class VectorStore:
    """Manage document embeddings in a ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf_documents_ollama",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for local Ollama RAG",
                    "hnsw:space": "cosine"
                }
            )

            print(
                f"vector store initialized with collection: "
                f"{self.collection_name}"
            )

            print(
                f"existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:

            print(
                f"Error initializing the vector store: {e}"
            )

            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):

        if len(documents) != len(embeddings):

            raise ValueError(
                "The number of documents and embeddings "
                "must be equal"
            )

        print(
            f"adding {len(documents)} documents "
            f"to vector store"
        )

        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            doc_id = (
                f"doc_{uuid.uuid4().hex[:8]}_{i}"
            )

            ids.append(doc_id)

            meta = (
                dict(doc.metadata)
                if hasattr(doc, "metadata")
                else {}
            )

            meta = {
                k: v
                for k, v in meta.items()
                if v is not None
            }

            for k, v in list(meta.items()):

                if not isinstance(
                    v,
                    (str, int, float, bool)
                ):
                    meta[k] = str(v)

            meta["doc_index"] = i
            meta["content_length"] = len(
                doc.page_content
            )

            metadatas.append(meta)

            documents_text.append(
                doc.page_content
            )

            embedding_list.append(
                embedding.tolist()
            )

        try:

            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"successfully added "
                f"{len(documents)} documents"
            )

            print(
                f"total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:

            print(
                f"error adding documents "
                f"to vector store: {e}"
            )

            raise

In [9]:
chunks

[Document(metadata={'producer': 'pdfFactory Pro 4.0 (Windows XP Professional)', 'creator': 'pdfFactory Pro www.pdffactory.com', 'creationdate': '2011-09-12T13:31:08+04:30', 'title': 'ensan dar jostojuoye mana', 'author': 'heydari', 'source': '..\\data\\pdf\\doctorfrankel.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'doctorfrankel.pdf', 'file_type': 'pdf'}, page_content='١ \naspx.151-post/com.blogfa.aarmaan://http  \n  \nا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن \n ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  \n وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  \n ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  \n  \n \nﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت \nﮔﻮردو دﮐﺘﺮ ﮐﻪ درآﻣﺪي ﭘﯿﺶ ﺑﺎ ﻣﻘﺪﻣﻪ ﻧﺪارد ﺟﺎ اﺳﺖ ﻧﻮﺷﺘﻪ ﮐﺘﺎب اﯾﻦ ﺑﺮ ﻫﺎروارد داﻧﺸﮕﺎه رواﻧﺸﻨﺎﺳﯽ ﭘﯿﺸﯿﻦ اﺳﺘﺎد آﻟﭙﻮرت ن\nﻣ را ﺗﻮﺿﯿﺤﺎﺗﯽ ﺗﻨﻬﺎ ﻣﻦ و ﺷﻮد ﻧﻮﺷﺘﻪ دﯾﮕﺮي ﯽدراز  ﻧﯿﺴﺖ ﻧﻮﺷﺘﻪ آن در ﮐﻪ اﻓﺰاﯾﻢ  . \n ﯾﻌﻨﯽ ﮐﺘﺎب اﯾﻦ اول اﺳﯿﺮان اردوي در ﺳﺮﮔﺸﺘﯽ «   را آن ﻣـﻦ و اﺳـﺖ ﻧﻮﺷﺘﻪ آﻟﻤﺎﻧﯽ ﺑﺰﺑﺎن ﻓﺮاﻧﮑﻞ دﮐﺘﺮ را  ﺗﺮﺟﻤـﻪ از\nام درآورده ﺑﻔﺎرﺳﯽ ﻻش اﯾﻠﺰه اﻧﮕﻠﯿﺴﯽ . ﯾﻌﻨﯽ دﯾﮕﺮ ﺑﺨﺶ ﻟﻮﮔﻮﺗﺮاﭘﯽ اﺳﺎﺳﯽ ﻣﻔﻬﻮم اﺳﺖ ﻧﻮﺷﺘﻪ ﺑﺎﻧﮕﻠﯿﺴﯽ ﻓﺮاﻧﮑﻞ را 

In [16]:
texts = [doc.page_content for doc in chunks]
texts

['١ \naspx.151-post/com.blogfa.aarmaan://http  \n  \nا ﻣﻌﻨﯽ ﺟﺴﺘﺠﻮي در ﻧﺴﺎن \n ﻧﻮﺷﺘﻪ : ﻓﺮاﻧﮑﻞ وﯾﮑﺘﻮر دﮐﺘﺮ  \n وﯾﻦ داﻧﺸﮕﺎه ﭘﺰﺷﮑﯽ روان اﺳﺘﺎد  \n ﺗﺮﺟﻤﻪ : ﻣﻌﺎرﻓﯽ اﮐﺒﺮ دﮐﺘﺮ  \n  \n \nﯿﺗﻮﺿ ﻓﺎرﺳﯽ ﻣﺘﺮﺟﻢ  ﺤﺎت \nﮔﻮردو دﮐﺘﺮ ﮐﻪ درآﻣﺪي ﭘﯿﺶ ﺑﺎ ﻣﻘﺪﻣﻪ ﻧﺪارد ﺟﺎ اﺳﺖ ﻧﻮﺷﺘﻪ ﮐﺘﺎب اﯾﻦ ﺑﺮ ﻫﺎروارد داﻧﺸﮕﺎه رواﻧﺸﻨﺎﺳﯽ ﭘﯿﺸﯿﻦ اﺳﺘﺎد آﻟﭙﻮرت ن\nﻣ را ﺗﻮﺿﯿﺤﺎﺗﯽ ﺗﻨﻬﺎ ﻣﻦ و ﺷﻮد ﻧﻮﺷﺘﻪ دﯾﮕﺮي ﯽدراز  ﻧﯿﺴﺖ ﻧﻮﺷﺘﻪ آن در ﮐﻪ اﻓﺰاﯾﻢ  . \n ﯾﻌﻨﯽ ﮐﺘﺎب اﯾﻦ اول اﺳﯿﺮان اردوي در ﺳﺮﮔﺸﺘﯽ «   را آن ﻣـﻦ و اﺳـﺖ ﻧﻮﺷﺘﻪ آﻟﻤﺎﻧﯽ ﺑﺰﺑﺎن ﻓﺮاﻧﮑﻞ دﮐﺘﺮ را  ﺗﺮﺟﻤـﻪ از\nام درآورده ﺑﻔﺎرﺳﯽ ﻻش اﯾﻠﺰه اﻧﮕﻠﯿﺴﯽ . ﯾﻌﻨﯽ دﯾﮕﺮ ﺑﺨﺶ ﻟﻮﮔﻮﺗﺮاﭘﯽ اﺳﺎﺳﯽ ﻣﻔﻬﻮم اﺳﺖ ﻧﻮﺷﺘﻪ ﺑﺎﻧﮕﻠﯿﺴﯽ ﻓﺮاﻧﮑﻞ را  . \n در ﮐﻪ ﮐﺘﺎب اﯾﻦ1963  ﻋﻨﻮان ﺑﺎMan’s search for Meaning           ﺑـﺰودي ﮐـﻪ ﯾﺎﻓـﺖ ﺷـﻬﺮﺗﯽ ﭼﻨـﺎن ﺷـﺪ ﻣﻨﺘﺸـﺮ\nرﺳﯿﺪ ﭼﺎپ ﺑﭽﻨﺪﯾﻦ. ﻧﺴﺨﻪ ﮐﺮدم ﺗﺮﺟﻤﻪ آن از ﻣﻦ ﮐﻪ اي  ﺳﺎل در و اﺳﺖ ﯾﺎزدﻫﻢ ﭼﺎپ1967 ﺷﺪه ﻣﻨﺘﺸﺮ  اﺳﺖ  . \n ﺑﺴﺎل ﻓﺮاﻧﮑﻞ دﮐﺘﺮ1905  در ﺷﻬﺮ ﻫﻤﺎن در و آﻣﺪ ﺑﺪﻧﯿﺎ وﯾﻦ در1930     در و رﺳـﺎﻧﯿﺪ ﺑﭙﺎﯾـﺎن را ﭘﺰﺷـﮑﯽ داﻧﺸﮑﺪه1942   ﮐـﻪ',
 'ﺑﺴﺎل ﻓﺮاﻧﮑﻞ دﮐﺘﺮ1905  در ﺷﻬﺮ ﻫﻤﺎن در و آﻣﺪ ﺑﺪﻧﯿﺎ وﯾﻦ در1930     در و ر

In [17]:
# اسم متغیر رو با اسم کلاس یکی نذار تا کلاس shadow نشه
vector_store = VectorStore()
vector_store

vector store initialized with collection: pdf_documents_ollama
existing documents in collection: 384


In [18]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embedding(
    texts
)

vector_store.add_documents(
    chunks,
    embeddings
)

generating Ollama embeddings for 384 texts...
generated embeddings with shape: (384, 768)
adding 384 documents to vector store
successfully added 384 documents
total documents in collection: 768


### RETRIEVER PIPELINE FROM VECTORTOR STORE

In [19]:
from typing import List, Dict, Any


class RAGRetriever:

    def __init__(
        self,
        vector_store,
        embedding_manager
    ):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(
            f"retrieving documents for: {query}"
        )

        print(
            f"top k: {top_k}, "
            f"score threshold: {score_threshold}"
        )

        # Generate query embedding with Ollama
        query_embedding = (
            self.embedding_manager
            .generate_query_embedding(query)
        )

        try:

            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )

            retrieved_docs = []

            if (
                results["documents"]
                and results["documents"][0]
            ):

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    doc_text,
                    meta,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Chroma cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({

                            "document": doc_text,

                            "metadata": meta,

                            "content": doc_text,

                            "id": doc_id,

                            "similarity_score":
                                similarity_score,

                            "distance": distance,

                            "rank": i + 1
                        })

                print(
                    f"retrieved "
                    f"{len(retrieved_docs)} documents"
                )

            else:

                print("no documents found")

            return retrieved_docs

        except Exception as e:

            print(
                f"error during retrieval: {e}"
            )

            return []

In [20]:

rag_retriever = RAGRetriever(vector_store, embedding_manager)
print("retriever ready!")

retriever ready!


In [21]:
docs = rag_retriever.retrieve(
    "معنا در زندگی چیست؟",
    top_k=3
)

for d in docs:

    print(
        d["rank"],
        d["similarity_score"],
        d["content"][:200]
    )

retrieving documents for: معنا در زندگی چیست؟
top k: 3, score threshold: 0.0
retrieved 3 documents
1 0.712352991104126 ﻣ قاﻃﻼ آﻧﭽﯿﺰي ﺑﻪ ﮐﻪ اﺳﺖ ﯽﻟﻮﮔﻮﺗﺮاﭘﯽ      اﺳـﺖ طﻣﺮﺑـﻮ ﻓـﺮد روﺣـﺎﻧﯽ وﺟـﻮد ﺑﺎ ﮐﻪ ﺷﻮد      . واژه ﻟﻮﮔـﻮﺗﺮاﭘﯽ در ﮐـﻪ داﺷـﺖ ﻧﻈـﺮ در ﺑﺎﯾـﺪ
روﺣﺎﻧﯽ ﻣ ﺑﮑﺎر ﻣﺬﻫﺒﯽ اﻣﻮر در ﯽﺑﺪاﻧﺴﺎﻧﮑﻪ ﻣ قاﻃﻼ اﻧﺴﺎﻧﯽ ﺑﺎﺑﻌﺎد ﺺﺑﺎﻻﺧ و ﮔ
2 0.7099924683570862 ﻗﺎﺑ آن ﻓﻠﺴﻔﯽ ﻓﻬﻢ ﺑﺎ ﻧﯿﺴﺖآزاد درﻣﺎن ﻞ. دارد وﺟﻮد ﻣﻮارد اﯾﻦ ﺑﺎ ﺷﺪن ﻣﻮاﺟﻪ ﺑﺮاي ﺑﺨﺼﻮﺻﯽ روﺷﻬﺎي ﻟﻮﮔﻮﺗﺮاﭘﯽ در اﻣﺎ  . 
ﻣ ﺑﮑﺎر روﺷﻬﺎ اﯾﻦ وﻗﺘﯽ ﺑﻔﻬﻤﯿﻢ آﻧﮑﻪ ﯽﺑﺮاي ﻣ ﺻﻮرت اﻋﻤﺎﻟﯽ ﭼﻪ ﯽرود ﻣ اﻧﺘﺨﺎب را ﻣﻮردي ﯽﮔﯿﺮد    
3 0.7005499601364136 ﻟﻮﮔﻮﺗﺮاﭘﯽ اﺳﺎﺳﯽ ﻣﻔﻬﻮم 
  
ﺳﺮﮔﺬ اﯾﻦ ﺧﻮاﻧﻨﺪﮔﺎن  ﻫﺴـﺘﻨﺪ ﺟﻮﯾـﺎ ﻣﻦ درﻣﺎﻧﯽ اﺻﻮل از ﮐﺎﻣﻠﺘﺮي و ﺢﺻﺮﯾ حﺷﺮ ﮐﻮﺗﺎه، ﺷﺖ       . درﺑـﺎره ﮐﻮﺗـﺎﻫﯽ ﻓﺼـﻞ ﻣـﻦ اﯾﻨﺠﻬـﺖ ﺑـﻪ
 اﺻﻠﯽ ﻧﻮﺷﺘﻪ در اﮔﺰﯾﺴﺘﺎﻧﺴﯿﺎﻟﯿﺰم ﺗﺎ ﻣﺮگ اردوﮔﺎه از 


### LOCAL DEPLOY 

In [22]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma4:12b",
    temperature=0
)

print("Ollama LLM ready!")

Ollama LLM ready!


In [23]:
def build_context(docs):

    if not docs:
        return "هیچ اطلاعات مرتبطی در اسناد پیدا نشد."

    context_parts = []

    for doc in docs:

        source = doc["metadata"].get(
            "source_file",
            "unknown"
        )

        page = doc["metadata"].get(
            "page",
            "unknown"
        )

        context_parts.append(
            f"""
Source: {source}
Page: {page}

Content:
{doc["content"]}
"""
        )

    return "\n\n".join(context_parts)

In [24]:
def ask_rag(
    question: str,
    top_k: int = 5
):

    # 1. Retrieve relevant documents
    docs = rag_retriever.retrieve(
        question,
        top_k=top_k
    )

    # 2. Build context
    context = build_context(docs)

    # 3. Create RAG prompt
    prompt = f"""
تو یک دستیار هوشمند هستی که باید فقط بر اساس
اطلاعات موجود در Context به سؤال کاربر پاسخ بدهی.

اگر پاسخ سؤال در Context وجود ندارد،
صادقانه بگو که اطلاعات کافی در اسناد موجود نیست.

Context:
-------------------------
{context}
-------------------------

Question:
{question}

Answer in Persian.
"""

    # 4. Generate answer locally with Ollama
    response = llm.invoke(prompt)

    # 5. Return everything useful
    return {
        "question": question,
        "answer": response.content,
        "documents": docs,
        "context": context
    }

In [29]:
result = ask_rag(
    "معنا در زندگی چیست؟",
    top_k=3
)

print("\nQUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for doc in result["documents"]:

    print(
        f"- {doc['metadata'].get('source_file', 'unknown')} "
        f"| page: {doc['metadata'].get('page', 'unknown')} "
        f"| score: {doc['similarity_score']:.4f}"
    )

retrieving documents for: معنا در زندگی چیست؟
top k: 3, score threshold: 0.0
retrieved 3 documents

QUESTION:
معنا در زندگی چیست؟

ANSWER:
اطلاعات کافی در اسناد موجود برای پاسخ به این سؤال وجود ندارد.

SOURCES:
- doctorfrankel.pdf | page: 34 | score: 0.7124
- doctorfrankel.pdf | page: 41 | score: 0.7100
- doctorfrankel.pdf | page: 32 | score: 0.7005


In [30]:
result = ask_rag(
    "معنا در زندگی چیست؟",
    top_k=3
)

print("========== RETRIEVED CONTEXT ==========\n")
print(result["context"])

retrieving documents for: معنا در زندگی چیست؟
top k: 3, score threshold: 0.0
retrieved 3 documents
========== RETRIEVED CONTEXT ==========


Source: doctorfrankel.pdf
Page: 34

Content:
ﻣ قاﻃﻼ آﻧﭽﯿﺰي ﺑﻪ ﮐﻪ اﺳﺖ ﯽﻟﻮﮔﻮﺗﺮاﭘﯽ      اﺳـﺖ طﻣﺮﺑـﻮ ﻓـﺮد روﺣـﺎﻧﯽ وﺟـﻮد ﺑﺎ ﮐﻪ ﺷﻮد      . واژه ﻟﻮﮔـﻮﺗﺮاﭘﯽ در ﮐـﻪ داﺷـﺖ ﻧﻈـﺮ در ﺑﺎﯾـﺪ
روﺣﺎﻧﯽ ﻣ ﺑﮑﺎر ﻣﺬﻫﺒﯽ اﻣﻮر در ﯽﺑﺪاﻧﺴﺎﻧﮑﻪ ﻣ قاﻃﻼ اﻧﺴﺎﻧﯽ ﺑﺎﺑﻌﺎد ﺺﺑﺎﻻﺧ و ﮔﺮدد ﻧﻤﯽ ﻣﺼﺮف ﯽرود ﺷﻮد.



Source: doctorfrankel.pdf
Page: 41

Content:
ﻗﺎﺑ آن ﻓﻠﺴﻔﯽ ﻓﻬﻢ ﺑﺎ ﻧﯿﺴﺖآزاد درﻣﺎن ﻞ. دارد وﺟﻮد ﻣﻮارد اﯾﻦ ﺑﺎ ﺷﺪن ﻣﻮاﺟﻪ ﺑﺮاي ﺑﺨﺼﻮﺻﯽ روﺷﻬﺎي ﻟﻮﮔﻮﺗﺮاﭘﯽ در اﻣﺎ  . 
ﻣ ﺑﮑﺎر روﺷﻬﺎ اﯾﻦ وﻗﺘﯽ ﺑﻔﻬﻤﯿﻢ آﻧﮑﻪ ﯽﺑﺮاي ﻣ ﺻﻮرت اﻋﻤﺎﻟﯽ ﭼﻪ ﯽرود ﻣ اﻧﺘﺨﺎب را ﻣﻮردي ﯽﮔﯿﺮد    ﻧﻮروﺗﯿﮑﻬـﺎ از ﺑﺴـﯿﺎري در ﮐﻪ ﮐﻨﯿﻢ



Source: doctorfrankel.pdf
Page: 32

Content:
ﻟﻮﮔﻮﺗﺮاﭘﯽ اﺳﺎﺳﯽ ﻣﻔﻬﻮم 
  
ﺳﺮﮔﺬ اﯾﻦ ﺧﻮاﻧﻨﺪﮔﺎن  ﻫﺴـﺘﻨﺪ ﺟﻮﯾـﺎ ﻣﻦ درﻣﺎﻧﯽ اﺻﻮل از ﮐﺎﻣﻠﺘﺮي و ﺢﺻﺮﯾ حﺷﺮ ﮐﻮﺗﺎه، ﺷﺖ       . درﺑـﺎره ﮐﻮﺗـﺎﻫﯽ ﻓﺼـﻞ ﻣـﻦ اﯾﻨﺠﻬـﺖ ﺑـﻪ
 اﺻﻠﯽ ﻧﻮﺷﺘﻪ در اﮔﺰﯾﺴﺘﺎﻧﺴﯿﺎﻟﯿﺰم ﺗﺎ ﻣﺮگ اردوﮔﺎه از آوردم . ﺧﻮاﺳـﺘﻨﺪ ﻣﻔﺼﻠﺘﺮي ﻧﻮﺷﺘﻪ و ﻧﻨﺸﺎﻧﺪ ﻓﺮو آﻧﺎﻧﺮا ﺗﺸﻨﮕﯽ ﻧﯿﺰ آ